**Notebook: RAG over the Saved News (Module 5)**
- OCBC AI Creator Masterclass, Day 1

___

This notebook is the hands-on part of Module 5 (about 32 minutes). After it come the example script
`day1/examples/m5_example.py` and the exercise `day1/exercises/m5_exercise.py`. Run it from the top.

# Background

The morning digest works. Now the RMs ask questions of their own: *"Why did the banks fall in mid October?"*,
*"What did NISP announce?"* The answers are in the news you saved in Module 2, but about 70 articles are too many to
paste into every prompt. With **RAG** the **question** decides which pieces of news reach the model:

- **5.1 Load the news.** Articles as documents with metadata.
- **5.2 Split.** Chunks small enough to be about one thing.
- **5.3 Embed.** Text as numbers that capture meaning.
- **5.4 Store and retrieve.** The chunks closest to a question.
- **5.5 Answer from the sources.** A grounded prompt with citations.

| Step on the ladder | Who decides WHICH data | Who decides WHEN |
|---|---|---|
| Report in the prompt (Modules 3–4) | Our code | The scheduler |
| **RAG (Module 5)** | **The user's question** | **The user, by asking** |

## Training Objectives

- Load saved articles as documents (plain dicts) with metadata
- Choose a chunk size and overlap, and see what they change
- Explain embeddings and cosine similarity, and compute a similarity yourself
- Retrieve the closest chunks with `k` and a metadata filter
- Write a grounded prompt that cites its sources and admits when they do not cover the question

## The Practice Data

Module 2 saved the news in `data/news/`. So that everyone retrieves the same articles, this notebook uses the copy
saved on 25 October 2026 in `data/sample/news_<TICKER>.json` (same format, 20 articles per bank). The example and the
exercise use your own `data/news/`.

The embedding model runs on your laptop: no key and no quota. The first use downloads it once (about 420 MB).

## Checking your answers

**Your turn** cells say what the right result is. Run the next cell once for the **Knowledge Check** questions.

In [ ]:
import hashlib

ANSWERS = {'m5_why': '16e529bd54', 'm5_overlap': 'edecb776fb', 'm5_embedding': '2de4008d1b', 'm5_filter': 'ac17981cf8', 'm5_debug': 'caa32026f7', 'm5_cover': 'd191bf7746'}

def check_answer(question, answer):
    key = hashlib.sha256(f"{question}:{answer.strip().lower()}".encode()).hexdigest()[:10]
    if answer.strip() == "":
        print("Type a, b, c or d between the quotes, then run the cell again.")
    elif key == ANSWERS[question]:
        print("Correct.")
    else:
        print("Not quite. Read the question again and try another answer.")

In [ ]:
import os
from dotenv import load_dotenv

# Work from the project root (the folder with data/), so file paths are the same as in the .py scripts
while not os.path.exists("data/sample"):
    os.chdir("..")

load_dotenv()  # reads the keys from .env
print("Working folder:", os.getcwd())

# 5.1 Load the News

> **The request**
>
> An RM: **"Why did the bank stocks fall so much in mid October?"** The answer is somewhere in 80 articles.

> **In this section**
>
> - **Key question:** Why not paste all the news into the prompt, and what does a document need to carry?
> - **How we run it:** **Live, about 5 minutes.**
> - **What you will do:** turn every article into a document (a plain dict) with metadata, and measure what pasting everything would cost.
> - **What carries forward:** `docs`, split in 5.2.

A document is a plain dict: the text in `"text"` and everything else in `"metadata"`. Metadata is what lets
us **filter** (by ticker) and **cite** (title, date) later. One article can mention two banks, so it can be in two
files: we keep each title once.

In [ ]:
import json

TICKERS = ["BBCA", "BBRI", "BMRI", "NISP"]

docs, seen_titles = [], set()
for ticker in TICKERS:
    with open(f"data/sample/news_{ticker}.json", encoding="utf-8") as f:
        for article in json.load(f):
            if article["title"] in seen_titles:
                continue
            seen_titles.add(article["title"])
            docs.append({
                "text": article["title"] + "\n\n" + article["body"],
                "metadata": {"title": article["title"], "date": article["timestamp"][:10],
                             "source": article["source"], "ticker": ticker},
            })

total_chars = sum(len(d["text"]) for d in docs)
print(len(docs), "articles,", f"{total_chars:,} characters (about {total_chars // 4:,} tokens) in every prompt")
docs[0]["metadata"]

#### Knowledge Check: Paste or retrieve?

When is pasting the data into the prompt better than RAG?

- **a.** Never
- **b.** When the data is large
- **c.** When the data is small and all of it is needed, like the Module 3 report
- **d.** When the user asks a question

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m5_why", answer)

# 5.2 Split into Chunks

> **The request**
>
> Your manager: **"A long article covers five topics. Retrieve the paragraph that answers, not the whole article."**

> **In this section**
>
> - **Key question:** How big should a chunk be?
> - **How we run it:** **Live, about 5 minutes.**
> - **What you will do:** split the articles, check that metadata travels with each chunk, and compare chunk sizes.
> - **What carries forward:** `chunks`, embedded in 5.4.

In [ ]:
def split_text(text, chunk_size=1500, chunk_overlap=100):
    """Cut a text into pieces of chunk_size characters; each piece repeats the last chunk_overlap characters of the one before."""
    chunks, start = [], 0
    while True:
        chunks.append(text[start:start + chunk_size])
        if start + chunk_size >= len(text):
            return chunks
        start += chunk_size - chunk_overlap


def split_documents(docs, chunk_size=1500, chunk_overlap=100):
    # Every piece keeps the metadata of its article
    return [{"text": piece, "metadata": doc["metadata"]}
            for doc in docs for piece in split_text(doc["text"], chunk_size, chunk_overlap)]


chunks = split_documents(docs)

print(len(docs), "articles ->", len(chunks), "chunks; longest", max(len(c["text"]) for c in chunks), "characters")
print(chunks[0]["metadata"])

#### Your turn: Compare chunk sizes

Split `docs` with `chunk_size` 500, 1500 and 4000 (overlap 10% of the size) and print, for each size, the number
of chunks and the average chunk length. Do **not** embed them. The right result: the smaller the size, the more chunks
(500 gives more than twice as many as 1500). all-mpnet-base-v2 reads at most 384 tokens (about 1,500 characters) of a
text and ignores the rest: which size fits?

In [ ]:
## Your code below



#### Knowledge Check: The overlap

What is the `chunk_overlap` for?

- **a.** It makes chunks smaller
- **b.** A sentence cut at a chunk border stays readable in both chunks
- **c.** It removes duplicate articles
- **d.** It speeds up the embedding

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m5_overlap", answer)

# 5.3 Embeddings and Similarity

> **The request**
>
> An RM types *"Why did bank stocks drop?"*, but the article says *"Foreign investors sold Indonesian bank
> shares"*. Not one word in common.

> **In this section**
>
> - **Key question:** How can a search find meaning instead of matching words?
> - **How we run it:** **Live, about 6 minutes.**
> - **What you will do:** embed three sentences and a question, then compute the similarity yourself.
> - **What carries forward:** `embedder`, used by the vector store in 5.4.

An **embedding** turns a text into a long list of numbers; texts with a similar **meaning** point in a similar
direction. **Cosine similarity** measures that: close to 1 means similar, close to 0 means unrelated. We use
`all-mpnet-base-v2`, an open-source embedding model that runs on your laptop. The embedder has one method for the texts
we store and one for the question (`embed_documents`, `embed_query`).

In [ ]:
import numpy as np
from pydantic_ai import Embedder

embedder = Embedder("sentence-transformers:all-mpnet-base-v2")   # open source, runs on your laptop

sentences = [
    "Foreign investors sold Indonesian bank shares after the MSCI announcement.",
    "Bank OCBC NISP completed an acquisition in the insurance business.",
    "The weather in Jakarta was sunny on Friday.",
]
result = await embedder.embed_documents(sentences)       # in a .py script: embedder.embed_documents_sync(...)
sentence_vectors = result.embeddings
question_vector = (await embedder.embed_query("Why did bank stocks drop?"))[0]
print(len(sentence_vectors), "vectors of", len(question_vector), "numbers")

#### Your turn: Compute the similarity

Write `cosine_similarity(a, b)` (the dot product divided by the product of the two lengths: `np.dot`, `np.linalg.norm`)
and print the score of the question against each sentence, highest first. The right result ranks the sell-off sentence
first and the weather sentence last. You will meet this function again in 5.4: the search scores every chunk with it.

In [ ]:
## Your code below



#### Knowledge Check: What the numbers mean

Two chunks have a cosine similarity of 0.92 with the question; a third has 0.41. What does that tell you?

- **a.** The first two contain more of the question's words
- **b.** The first two are closer in meaning to the question
- **c.** The third is newer
- **d.** The first two are longer

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m5_embedding", answer)

# 5.4 Store and Retrieve

> **The request**
>
> The RM's question again: **"Why did Indonesian bank stocks fall sharply in mid October 2026?"**

> **In this section**
>
> - **Key question:** How do we find the chunks that answer a question, and only from the right bank?
> - **How we run it:** **Live, about 7 minutes.**
> - **What you will do:** embed all chunks into a vector store, search with `k`, then add a metadata filter.
> - **What carries forward:** `vector_store`, which `ask_news` uses in 5.5.

Our **vector store** is just the chunks and their vectors side by side. A search embeds the question, scores every
chunk with cosine similarity (the function you wrote in 5.3, repeated below so this cell runs on its own) and keeps
the `k` closest. That is enough for our news (about 80 chunks, embedded once on the laptop). A `filter` on the
metadata keeps only some chunks.

In [ ]:
def cosine_similarity(a, b):      # the same function as in 5.3
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


async def build_vector_store(chunks):
    result = await embedder.embed_documents([c["text"] for c in chunks])
    return {"chunks": chunks, "vectors": np.array(result.embeddings)}


async def similarity_search(vector_store, query, k=4, filter=None):
    query_vector = (await embedder.embed_query(query))[0]
    scored = [(cosine_similarity(query_vector, v), c) for v, c in zip(vector_store["vectors"], vector_store["chunks"])]
    scored = [(s, c) for s, c in scored if filter is None or filter(c)]
    return [c for s, c in sorted(scored, key=lambda pair: pair[0], reverse=True)[:k]]


vector_store = await build_vector_store(chunks)

question = "Why did Indonesian bank stocks fall sharply in mid October 2026?"
for doc in await similarity_search(vector_store, question, k=4):
    print(doc["metadata"]["date"], "|", doc["metadata"]["ticker"], "|", doc["metadata"]["title"][:80])

#### Your turn: Only one bank, only recent news

An RM covering NISP asks *"What did the bank announce about acquisitions?"* Retrieve the **3** closest chunks, but
only from **NISP** news **dated 2026-10-01 or later** (the `filter` is a function that gets a chunk dict and returns
`True` to keep it; dates are `YYYY-MM-DD` strings, so `>=` works). Print date, ticker and title. The right result is
3 NISP chunks, all from October 2026.

In [ ]:
## Your code below



#### Knowledge Check: The wrong bank

A question about BBRI gets an answer built on BBCA news. What is the simplest fix?

- **a.** A filter on the ticker
- **b.** A bigger model
- **c.** Longer chunks
- **d.** A higher `k`

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m5_filter", answer)

# 5.5 Answer from the Sources

> **The request**
>
> Your manager: **"An RM must be able to check every sentence. And if the news does not say, the answer must say so."**

> **In this section**
>
> - **Key question:** How do we make the answer grounded and checkable?
> - **How we run it:** **Live, about 8 minutes.**
> - **What you will do:** put the retrieved chunks in the prompt as numbered sources, ask with and without coverage, then extend `ask_news`.
> - **What carries forward:** `ask_news`, which becomes the `search_news` tool on Day 2.

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

model = OpenAIResponsesModel(
    os.getenv("OPENAI_MODEL", "gpt-6.1-sol"),
    provider=OpenAIProvider(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_URL")),
)
rag_agent = Agent(model)

RULES = """Answer the question using only the sources below. Cite them as [1], [2], ...
If the sources do not answer the question, say "The saved news does not cover this."
Never give buy or sell advice."""


async def ask_news(question, k=4):
    results = await similarity_search(vector_store, question, k=k)
    sources = ""
    for i, doc in enumerate(results, start=1):
        sources += f"[{i}] {doc['metadata']['title']} ({doc['metadata']['date']})\n{doc['text']}\n\n"
    result = await rag_agent.run(f"{RULES}\n\n<sources>\n{sources}</sources>\n\nQuestion: {question}")
    return result.output, results


answer, results = await ask_news(question)
print(answer, "\n")
for i, doc in enumerate(results, start=1):
    print(f"[{i}] {doc['metadata']['title']} ({doc['metadata']['date']})")

In [ ]:
answer, _ = await ask_news("What is BBCA's dividend policy for 2030?")      # not in the news
print(answer)

#### Your turn: Ask about one bank

Write `ask_bank_news(ticker, question, k=3)`: the same as `ask_news`, but it retrieves only chunks of that ticker
(your filter from 5.4) and returns the answer and the list of `(date, title)` of the sources. Ask it about **NISP**:
*"What did the bank announce about acquisitions?"* The right result cites its sources (`[1]`, `[2]`, ...), and every source is a NISP article.

In [ ]:
## Your code below



#### Knowledge Check: A poor answer

An RM says an answer missed an important article. What do you check **first**?

- **a.** The model: switch to a bigger one
- **b.** The retrieved chunks: was the article among them?
- **c.** The system prompt wording
- **d.** The chunk overlap

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m5_debug", answer)

#### Knowledge Check: Not covered

Why does the prompt tell the model to say "The saved news does not cover this."?

- **a.** To save tokens
- **b.** So the model does not fill a gap with its own guess, which an RM could take as fact
- **c.** Because the model provider requires it
- **d.** To make the answer shorter

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m5_cover", answer)

# Module 5 Wrap-up

| Step | Here | Decides |
|---|---|---|
| Load | a dict: `"text"` + `"metadata"` | what we can filter and cite |
| Split | `split_text` (chunk size, overlap) | what one search hit contains |
| Embed + store | `Embedder("sentence-transformers:all-mpnet-base-v2")`, chunks and vectors side by side | how meaning is compared |
| Retrieve | `similarity_search(vector_store, question, k, filter)`, cosine similarity | **which data the model sees** |
| Answer | numbered sources, rules | whether an RM can check it |

**Day 1 in one line:** *we tell the LLM what the data is.* Tomorrow `ask_news` becomes a tool, and the LLM asks for
data itself. **Next:** `day1/examples/m5_example.py`, then `day1/exercises/m5_exercise.py`.